# 04 — Aggregate

Joins the cleaned crime data to the community-area lookup and exports four CSVs under `data/tableau/`, each shaped for one dashboard view: per-area totals, a day-by-hour heatmap table, a monthly trend, and a sampled point layer for the density map.

In [ ]:
import pandas as pd

crimes = pd.read_parquet("data/crimes_clean.parquet")
lookup = pd.read_parquet("data/community_lookup.parquet")

# attach area name + socioeconomic context to every crime
df = crimes.merge(lookup, on="community_area", how="left")

print(f"{len(df):,} rows after join")
# check the join worked — should be very few/zero unmatched
print("Rows with no area name:", df["area_name"].isna().sum())
df[["community_area", "area_name", "crime_category", "per_capita_income"]].head()

In [ ]:
import os
os.makedirs("data/tableau", exist_ok=True)

## By area

Crime counts and arrest rate per community area and category, joined with the socioeconomic context — feeds the neighborhood ranking and income-vs-crime views.

In [ ]:
area_table = (
    df.groupby(["community_area", "area_name", "crime_category"])
      .agg(
          crimes=("id", "size"),
          arrests=("arrest", "sum"),
      )
      .reset_index()
)

# attach the socioeconomic context (one row per area, so take first)
context = df.groupby("community_area")[
    ["pct_below_poverty", "per_capita_income", "hardship_index"]
].first().reset_index()

area_table = area_table.merge(context, on="community_area", how="left")
area_table["arrest_rate"] = (area_table["arrests"] / area_table["crimes"]).round(3)

area_table.to_csv("data/tableau/crimes_by_area.csv", index=False)
print(f"crimes_by_area.csv — {len(area_table)} rows")
area_table.head()

## By time

Crime counts by day-of-week and hour — feeds the hour×day temporal heatmap. Days are ordered explicitly so Tableau doesn't sort them alphabetically.

In [ ]:
# order days properly so Tableau doesn't sort them alphabetically
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]

time_table = (
    df.groupby(["dayofweek", "hour"])
      .agg(crimes=("id", "size"))
      .reset_index()
)
time_table["dayofweek"] = pd.Categorical(time_table["dayofweek"],
                                         categories=day_order, ordered=True)
time_table = time_table.sort_values(["dayofweek", "hour"])

time_table.to_csv("data/tableau/crimes_by_time.csv", index=False)
print(f"crimes_by_time.csv — {len(time_table)} rows")  # ~168 (7 days x 24 hrs)
time_table.head()

## Trend

Monthly crime counts by category, for the time-series view.

In [ ]:
trend_table = (
    df.groupby(["year", "month", "crime_category"])
      .agg(crimes=("id", "size"))
      .reset_index()
)
# a real date for clean time-axis plotting in Tableau
trend_table["year_month"] = pd.to_datetime(
    trend_table["year"].astype(str) + "-" +
    trend_table["month"].astype(str) + "-01"
)

trend_table.to_csv("data/tableau/crimes_trend.csv", index=False)
print(f"crimes_trend.csv — {len(trend_table)} rows")
trend_table.head()

## Point sample

A 75k-row random sample of individual crimes for the density heatmap — enough to be visually convincing without shipping the full 1.5M-row dataset as CSV.

In [ ]:
# 75k random points is plenty for a convincing density map, keeps the file small
sample = df.sample(n=75_000, random_state=42)[
    ["latitude", "longitude", "crime_category", "year", "arrest"]
]
sample.to_csv("data/tableau/crime_points_sample.csv", index=False)

size_kb = os.path.getsize("data/tableau/crime_points_sample.csv") / 1e3
print(f"crime_points_sample.csv — {len(sample):,} rows ({size_kb:.0f} KB)")
print("\nAll Tableau files:", os.listdir("data/tableau"))